# 20 Audio Encoder Comparison

**Purpose:** Cache BEATs embeddings for the audio tasks and pick which encoder layer the audio heads read, with linear probes.

**Inputs:** `configs/pipeline.yaml` (audio encoder), BEATs weights `models/audio/beats/BEATs_iter3_plus_AS2M.pt`, `audio_emotion` and `audio_context` rows.

**Outputs:** Embedding caches in `<data dir>/cache/audio/`; the decision to use the mean over layers (`branches.audio.layer: mean`).

**Splits used:** train (probes, out-of-fold); val and test embeddings are cached only (test: cached, not scored).

- Embeddings are made one clip at a time (batch_size = 1)
- Batching changed them by up to 8%, so training embedding would not match the app's.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from woofdata.tasks import task_rows
from woofdecoder.audio import encoder_from_config
from woofdecoder.config import load_config
from wooftrainer.audio_trainer.embeddings import attach_labels, cache_embeddings, load_embeddings

pd.set_option("display.width", 200)

In [ ]:
SPLITS = [
    ("audio_emotion", "val"),
    ("audio_emotion", "train"),
    ("audio_context", "val"),
    ("audio_context", "test"),
    ("audio_context", "train"),
]
SEED = 0  # cross-validation shuffling

## Loading necessary project configurations

Load the configured audio encoder and print its settings.

In [2]:
encoder = encoder_from_config(load_config())
print(f"encoder name: {encoder.name}")
print(f"encoder device: {encoder.device}")
print(f"encoder num_layers: {encoder.num_layers}")
print(f"encoder dim: {encoder.dim}")
print(f"encoder settings: {encoder.settings}")

c:\Users\rlaeh\dev\dog-vocal-interpreter\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


encoder name: beats_iter3_plus_as2m
encoder device: cuda
encoder num_layers: 12
encoder dim: 768
encoder settings: EncoderSettings(min_seconds=1.0, chunk_seconds=10.0, batch_size=1)


## Caching resulting embeddings

Embed every clip of each task and split once and save the cache.

In [3]:
for task, split in SPLITS:
    start = time.perf_counter()

    cache_embeddings(task=task, split=split, encoder=encoder)

    print(f"Task: {task}, Split: {split} took {time.perf_counter() - start:.0f} s")

audio_emotion val: 200 clips, 200 cached, 0 to embed, 0 dropped
Task: audio_emotion, Split: val took 0 s
audio_emotion train: 1000 clips, 1000 cached, 0 to embed, 0 dropped
Task: audio_emotion, Split: train took 0 s
audio_context val: 441 clips, 403 cached, 38 to embed, 0 dropped
Task: audio_context, Split: val took 4 s
Reminder: test is reported once, at the end. Tune on val.
audio_context test: 358 clips, 357 cached, 1 to embed, 0 dropped
Task: audio_context, Split: test took 0 s
audio_context train: 13065 clips, 12980 cached, 85 to embed, 0 dropped
Task: audio_context, Split: train took 10 s


### Check if cache match the manifest

In [4]:
for task, split in SPLITS:
    embeddings, table = load_embeddings(task, split, encoder.name)
    rows = task_rows(task, split)
    same_clips = len(rows) == len(table) and set(rows["clip_id"]) == set(table["clip_id"])

    print(f"Task: {task:<14} Split: {split:<5} Shape: {embeddings.shape}")
    print(f"Matches Manifest: {same_clips} Finite: {np.isfinite(embeddings).all()}")

Task: audio_emotion  Split: val   Shape: (200, 12, 768)
Matches Manifest: True Finite: True
Task: audio_emotion  Split: train Shape: (1000, 12, 768)
Matches Manifest: True Finite: True
Task: audio_context  Split: val   Shape: (441, 12, 768)
Matches Manifest: True Finite: True
Reminder: test is reported once, at the end. Tune on val.
Task: audio_context  Split: test  Shape: (358, 12, 768)
Matches Manifest: True Finite: True
Task: audio_context  Split: train Shape: (13065, 12, 768)
Matches Manifest: True Finite: True


### Check labels

In [5]:
_, emotion = load_embeddings("audio_emotion", "train", encoder.name)
print(emotion["arousal"].value_counts(), "\n")
print(emotion["valence"].value_counts(), "\n")

_, context = load_embeddings("audio_context", "train", encoder.name)
print(context["mapped_category"].replace("", "(none)").value_counts(), "\n")

print("manifest columns:", list(task_rows("audio_emotion", "train").columns))

arousal
Medium    366
High      323
Low       311
Name: count, dtype: int64 

valence
Positive    429
Neutral     340
Negative    231
Name: count, dtype: int64 

mapped_category
playful_excited      5570
other                3157
attention_seeking    2429
alert_aggressive     1824
Name: count, dtype: int64 

manifest columns: ['clip_id', 'source', 'source_version', 'source_license', 'dog_id', 'leakage_group', 'split', 'modality', 'file_path', 'start_s', 'end_s', 'target_path', 'annotation_path', 'original_label', 'label_type', 'label_provenance', 'mapped_category', 'eval_allowed', 'notes']


## Layer Probe

LogisticRegression model in scikit-learn has input nodes -> output nodes, they are no hidden nodes. This means that it is a single neural network perceptron.

A probe is a quick test of what an embedding contains

A probe puts a very simple model, on top of frozen embedding to measure how much useful information they already hold.

### Load the Embeddings with Current Labels

In [7]:
emotion_x, emotion = load_embeddings("audio_emotion", "train", encoder.name)
emotion = attach_labels(emotion, "audio_emotion", "train")
context_x, context = load_embeddings("audio_context", "train", encoder.name)
context = attach_labels(context, "audio_context", "train")

NUM_LAYERS = emotion_x.shape[1]
CANDIDATES = {"layer 3": 2, "layer 7": 6, f"layer {NUM_LAYERS}": NUM_LAYERS - 1, "mean": "mean"}

print("emotion", emotion_x.shape, "| context", context_x.shape, "| candidates", list(CANDIDATES))

emotion (1000, 12, 768) | context (12980, 12, 768) | candidates ['layer 3', 'layer 7', 'layer 12', 'mean']


### Helpers

In [8]:
def layer_features(embeddings, layer):
    if layer == "mean":
        return embeddings.mean(axis=1)
    return embeddings[:, layer]


def out_of_fold_probs(features, labels, cv, groups=None):
    model = make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)
    )
    return cross_val_predict(
        model, features, labels, cv=cv, groups=groups, method="predict_proba", n_jobs=-1
    )


def macro_f1(labels, predicted):
    return f1_score(labels, predicted, average="macro")


def bootstrap_ci(labels, predicted, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(repeats):
        index = rng.integers(0, len(labels), len(labels))
        scores.append(macro_f1(labels[index], predicted[index]))
    return np.percentile(scores, [2.5, 97.5])


def score(task_name, candidate, labels, probs):
    classes = np.unique(labels)
    predicted = classes[probs.argmax(axis=1)]
    low, high = bootstrap_ci(labels, predicted)
    return {
        "task": task_name,
        "candidate": candidate,
        "macro_f1": macro_f1(labels, predicted),
        "ci_low": low,
        "ci_high": high,
        "log_loss": log_loss(labels, probs, labels=classes),
        "predicted": predicted,
    }

### Emotion Probe

In [9]:
results = []
emotion_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, layer in CANDIDATES.items():
    features = layer_features(emotion_x, layer)
    for target in ("arousal", "valence"):
        labels = emotion[target].to_numpy()
        probs = out_of_fold_probs(features, labels, emotion_cv)
        results.append(score(f"emotion {target}", name, labels, probs))
    print("done:", name)

done: layer 3
done: layer 7
done: layer 12
done: mean


### Context Probe

In [ ]:
context_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
context_labels = context["mapped_category"].to_numpy()
context_groups = context["dog_id"].to_numpy()

for name, layer in CANDIDATES.items():
    start = time.perf_counter()
    probs = out_of_fold_probs(
        layer_features(context_x, layer), context_labels, context_cv, context_groups
    )
    result = score("context", name, context_labels, probs)
    result["distressed_f1"] = np.atleast_1d(
        f1_score(context_labels, result["predicted"], labels=["distressed_lonely"], average=None)
    )[0]

    results.append(result)
    print(f"done: {name} ({time.perf_counter() - start:.0f}s)")

### Table

In [11]:
table = pd.DataFrame(results).drop(columns="predicted")
table = table.set_index(["task", "candidate"]).round(3)
print(table.to_string())

                           macro_f1  ci_low  ci_high  log_loss  distressed_f1
task            candidate                                                    
emotion arousal layer 3       0.466   0.434    0.497     2.742            NaN
emotion valence layer 3       0.384   0.353    0.415     3.331            NaN
emotion arousal layer 7       0.488   0.459    0.518     2.633            NaN
emotion valence layer 7       0.390   0.359    0.423     3.272            NaN
emotion arousal layer 12      0.475   0.443    0.503     2.719            NaN
emotion valence layer 12      0.385   0.356    0.416     3.272            NaN
emotion arousal mean          0.513   0.482    0.543     2.412            NaN
emotion valence mean          0.423   0.390    0.454     3.171            NaN
context         layer 3       0.380   0.373    0.388     1.386            0.0
                layer 7       0.400   0.391    0.408     1.348            0.0
                layer 12      0.409   0.400    0.417     1.336  

## Results and decisions

- Context has 5 classes thus log loss of blind guess is equal to ln(5) which is around 1.61
- Emotion has 3 classes thus log loss of blind guess is equal to ln(3) which is around 1.10

Default is layer 12 vs mean on emotion and mean wins clearly on both targets.
- Arousal macro F1: 0.475 for layer 12, 0.514 for mean
- Valence macro F1: 0.384 for layer 12, 0.425 for mean

Layer 3 and 7 don't replace mean beause both emotion CI lower bounds above mean score.

**Valence is weak, which 0.43 against 0.33 for gusessing among 3 classes is only a little better than chance.**
- For the nudge, emotion signal will add a small hint to the overall output of audio branch.
- For extras, the app should present valence as weak hint. 

**Distressed F1 of 0.5-0.65 is suspiciously good, in which it is 62 windows of about 16 dogs against 13,000 Act_Env clips. (only distressed_lonely in self_labeled)** 
- Probably, youtube sound from Act_Env sound shows hint during training (microphones, compression, background)
- Source leakage (if head calls most of non-distressed fusion train data as distressed_lonely) -> will add non-distressed youtube clips.

**We will use mean layers for the heads.**